Etapa 1 — Preparar o ambiente

In [ ]:
from pathlib import Path 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

URL = (
    "https://raw.githubusercontent.com/IBM/"
    "telco-customer-churn-on-icp4d/master/data/"
    "Telco-Customer-Churn.csv"
)
arquivo = Path("data/Telco-Customer-Churn.csv")
arquivo.parent.mkdir(parents=True, exist_ok=True)

if not arquivo.exists():
    dados = pd.read_csv(URL)
    dados.to_csv(arquivo, index=False)

df = pd.read_csv(arquivo)

print("Dimensões:", df.shape)
display(df.head())


Etapa 2 — Obter e conhecer os dados

In [ ]:
print(df.info())

print("\nDistribuição de cancelamentos:")
print(df["Churn"].value_counts(normalize=True))

print("\nValores ausentes:")
print(df.isna().sum().sort_values(ascending=False).head(10))

print("\nClientes duplicados:")
print(df["customerID"].duplicated().sum())

Etapa 3 — Preparar os dados e separar treinamento de teste

In [ ]:
from sklearn.model_selection import train_test_split

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Variável categórica codificada originalmente como 0/1
df["SeniorCitizen"] = df["SeniorCitizen"].astype(str)

# Alvo: cancelou = 1; permaneceu = 0
y = df["Churn"].map({"Yes": 1, "No": 0})

# Não utilizar identificador ou variável-alvo como atributo
X = df.drop(columns=["customerID", "Churn"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treinamento:", len(X_train))
print("Teste:", len(X_test))

assert y.notna().all()
assert set(X_train.index).isdisjoint(set(X_test.index))
assert "Churn" not in X_train.columns
assert "customerID" not in X_train.columns


Etapa 4 — Construir o baseline

In [ ]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

pred_baseline = baseline.predict(X_test)

print("Acurácia:", accuracy_score(y_test, pred_baseline))
print("Precisão:", precision_score(
    y_test, pred_baseline, zero_division=0
))
print("Recall:", recall_score(
    y_test, pred_baseline, zero_division=0
))
print("F1:", f1_score(
    y_test, pred_baseline, zero_division=0
))


Etapa 5 — Treinar um modelo real de Machine Learning

In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.linear_model import LogisticRegression

numericas = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categoricas = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

processamento = ColumnTransformer([
    ("num", Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler())
    ]), numericas),

    ("cat", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore"))
    ]), categoricas)
])

modelo = Pipeline([
    ("preparar", processamento),
    ("classificar", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

modelo.fit(X_train, y_train)

pred_ml = modelo.predict(X_test)
proba_ml = modelo.predict_proba(X_test)[:, 1]


Etapa 6 — Comparar baseline e modelo

In [ ]:

def metricas(y_real, y_previsto):
    return {
        "Acurácia": accuracy_score(y_real, y_previsto),
        "Precisão": precision_score(
            y_real, y_previsto, zero_division=0
        ),
        "Recall": recall_score(
            y_real, y_previsto, zero_division=0
        ),
        "F1": f1_score(
            y_real, y_previsto, zero_division=0
        )
    }

resultados = pd.DataFrame({
    "Baseline": metricas(y_test, pred_baseline),
    "Regressão logística": metricas(y_test, pred_ml)
}).T

display(resultados.round(3))


In [ ]:

ax = resultados.plot(
    kind="bar",
    figsize=(10, 5),
    rot=0
)

ax.set_title("Baseline vs. Machine Learning")
ax.set_ylabel("Valor da métrica")
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()


Etapa 7 — Analisar os erros

In [ ]:

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)

print(classification_report(
    y_test,
    pred_ml,
    target_names=["Permaneceu", "Cancelou"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_ml,
    display_labels=["Permaneceu", "Cancelou"],
    cmap="Blues"
)

plt.title("Matriz de confusão — Regressão logística")
plt.tight_layout()
plt.show()


In [ ]:

analise = X_test.copy()

analise["Real"] = y_test
analise["Previsto"] = pred_ml
analise["Probabilidade_cancelamento"] = proba_ml

erros = analise[
    analise["Real"] != analise["Previsto"]
]

colunas = [
    "tenure",
    "Contract",
    "MonthlyCharges",
    "Real",
    "Previsto",
    "Probabilidade_cancelamento"
]

display(erros[colunas].head(10))


Etapa 8 — Criar testes automatizados

In [ ]:
from pathlib import Path
import pandas as pd

ARQUIVO = Path("data/Telco-Customer-Churn.csv")

print("Arquivo encontrado:", ARQUIVO.exists())
def test_arquivo_existe():
    assert ARQUIVO.exists(), (
        "Execute primeiro a etapa de download dos dados."
    )

def test_colunas_obrigatorias():
    df = pd.read_csv(ARQUIVO)

    colunas = {
        "customerID",
        "tenure",
        "MonthlyCharges",
        "Churn"
    }

    assert colunas.issubset(df.columns)

def test_identificadores_unicos():
    df = pd.read_csv(ARQUIVO)

    assert df["customerID"].is_unique

def test_classes_validas():
    df = pd.read_csv(ARQUIVO)

    assert set(df["Churn"].unique()) == {"Yes", "No"}


In [ ]:
import sys
import pandas as pd
import sklearn

print("Python:", sys.version)
print("Executável:", sys.executable)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)